In [1]:
#Import libraries
import numpy as np
import pandas as pd
import os

In [2]:
# Read the CSV file
data = pd.read_csv('Lifespan/companyDrug/speeds_and_coordinates_20241016_3_updated.csv')

In [3]:
# Display the first few rows of the dataset
print(data.head())
#Summary of data
print(data.describe())

   Frame       Speed           X           Y  Changed Pixels
0      1   14.152425  476.356932  487.840708             460
1      2   21.822664  419.305000  511.537500             494
2      3   22.484239  430.031250  509.536318             200
3      4  120.411300  453.283636  505.876364             310
4      5  103.131087  414.000000  551.500000             200
              Frame         Speed             X             Y  Changed Pixels
count  75593.000000  75593.000000  74855.000000  74855.000000    75593.000000
mean    5400.000000      1.420182    347.420578    497.622615       23.145662
std     3117.423385     11.178094    121.705676    129.644571       42.538445
min        1.000000      0.000000    124.005291    159.162088        0.000000
25%     2700.000000      0.000000    249.765571    390.323443        0.000000
50%     5400.000000      0.275513    336.299479    513.923404        8.000000
75%     8100.000000      1.076040    425.747680    632.201238       30.000000
max    107

In [4]:
#Correct Frame numbers --> now all frames are unique, use as ID column
data["continuous_frames"] = data.index + 1


In [5]:
print(data.shape)
print(f"Total time recorded: {round(data.shape[0]*2/3600,3)} hours")

(75593, 6)
Total time recorded: 41.996 hours


In [6]:
#Add Euclidean Speed to the next column
data['euclidean_speed'] = np.sqrt((data['X'].diff())**2 + (data['Y'].diff())**2) * 2
data['euclidean_speed'] = data['euclidean_speed'].shift(-1)

data['speed_diff'] = data['euclidean_speed'] - data['Speed']
data['speed_diff'] = np.where(abs(data['speed_diff']) < 0.01, 0, data['speed_diff'])

In [7]:
# print(data[['euclidean_speed', 'Speed', 'speed_diff']])
non_zero_count = (data['speed_diff'] != 0).sum()

print(f"Number of rows where speed_diff is not 0: {non_zero_count}")

Number of rows where speed_diff is not 0: 1172


In [8]:
# Step 1: Check for "no significant movement" with a tolerance of 0.5
tolerance_movement = 0.5
# data['no_movement'] = (data['X'].diff().fillna(0).abs() < tolerance_movement) & (data['Y'].diff().fillna(0).abs() < tolerance_movement)
tolerance_speed = 0.1
data['no_movement'] = data['Speed'].abs() < tolerance_speed

# Step 2: Count consecutive no-movement frames
# Use a cumulative counter that resets when movement is detected
data['no_movement_count'] = data['no_movement'].cumsum() - data['no_movement'].cumsum().where(~data['no_movement']).ffill().fillna(0).astype(int)

# Step 3: Identify when no movement reaches the threshold
threshold = 153
data['dead'] = data['no_movement_count'] >= threshold

# Step 4: Record the first row where the worm is declared dead
dead_row = data[data['dead']].index.min()

if pd.notna(dead_row):
    print(f"The worm is declared dead at row index: {dead_row}")
    # Display relevant columns for verification
    print(data[['Frame', 'X', 'Y', 'Changed Pixels', 'Speed', 'no_movement', 'no_movement_count', 'dead']].iloc[max(dead_row - 3, 0):min(dead_row + 3, len(data) - 1)])
else:
    print("The worm never reaches the dead threshold.")



The worm is declared dead at row index: 4814
      Frame           X           Y  Changed Pixels     Speed  no_movement  \
4811   4812         NaN         NaN               0  0.000000         True   
4812   4813         NaN         NaN               0  0.000000         True   
4813   4814         NaN         NaN               0  0.000000         True   
4814   4815         NaN         NaN              13  0.000000         True   
4815   4816  334.000000  628.000000              13  1.233171        False   
4816   4817  333.384615  628.038462               7  0.408005        False   

      no_movement_count   dead  
4811                150  False  
4812                151  False  
4813                152  False  
4814                153   True  
4815                  0  False  
4816                  0  False  


In [11]:
from exploration_trial import worm_death

# Iterate over all CSV files in the "Lifespan/companyDrug" directory
directory_path = "Lifespan/companyDrug"

worm_death_dict = {}

for file in os.listdir(directory_path):
    if file.endswith(".csv"):
        filepath = os.path.join(directory_path, file)
        
        # Extract worm ID from the filename
        worm_id = file.split("_")[4]  # Adjust the index based on the filename format
        
        # Call the worm_death function for each file
        dead_row = worm_death(filepath)
    
        worm_death_dict[worm_id] = [dead_row, 1] #1 indicates drug
        
        if pd.notna(dead_row):
            print(f"Worm {worm_id} is declared dead at row index {dead_row}.")
        else:
            print(f"Worm {worm_id} never reaches the dead threshold.")

directory_path = "Lifespan/control"
for file in os.listdir(directory_path):
    if file.endswith(".csv"):
        filepath = os.path.join(directory_path, file)
        
        # Extract worm ID from the filename
        worm_id = file.split("_")[4]  # Adjust the index based on the filename format
        
        # Call the worm_death function for each file
        dead_row = worm_death(filepath)
    
        worm_death_dict[worm_id] = [dead_row, 0] #0 indicates no drug
        
        if pd.notna(dead_row):
            print(f"Worm {worm_id} is declared dead at row index {dead_row}.")
        else:
            print(f"Worm {worm_id} never reaches the dead threshold.")

Worm 24 never reaches the dead threshold.
Worm 3 is declared dead at row index 4761.
Worm 6 is declared dead at row index 1156.
Worm 21 is declared dead at row index 3578.
Worm 22 never reaches the dead threshold.
Worm 5 never reaches the dead threshold.
Worm 2 is declared dead at row index 842.
Worm 25 is declared dead at row index 7154.
Worm 26 is declared dead at row index 45994.
Worm 1 never reaches the dead threshold.
Worm 4 is declared dead at row index 99.
Worm 23 is declared dead at row index 25353.
Worm 16 is declared dead at row index 5826.
Worm 31 is declared dead at row index 99.
Worm 34 is declared dead at row index 13281.
Worm 13 is declared dead at row index 998.
Worm 32 is declared dead at row index 99.
Worm 15 never reaches the dead threshold.
Worm 12 is declared dead at row index 7071.
Worm 35 is declared dead at row index 852.
Worm 14 is declared dead at row index 99.
Worm 33 is declared dead at row index 1753.
Worm 36 is declared dead at row index 59668.
Worm 11 is 

In [12]:
print(worm_death_dict)

{'24': [nan, 1], '3': [4761, 1], '6': [1156, 1], '21': [3578, 1], '22': [nan, 1], '5': [nan, 1], '2': [842, 1], '25': [7154, 1], '26': [45994, 1], '1': [nan, 1], '4': [99, 1], '23': [25353, 1], '16': [5826, 0], '31': [99, 0], '34': [13281, 0], '13': [998, 0], '32': [99, 0], '15': [nan, 0], '12': [7071, 0], '35': [852, 0], '14': [99, 0], '33': [1753, 0], '36': [59668, 0], '11': [2798, 0]}
